# 01. 탐색적 데이터 분석 (EDA)
담당: 데이터 A · 함수는 `src/eda.py`, `src/visualize.py`에 있고 이 노트북은 결과를 보고 해석만 합니다.

각 섹션 아래 **인사이트** 칸에 발견한 점과 다음 행동(정제·증강·모델 실험)을 적어 주세요. 보고서에 그대로 옮깁니다.

In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd

from src.utils import load_config
from src.visualize import set_korean_font

set_korean_font()
pd.set_option("display.max_columns", 50)

CONFIG = "configs/base.yaml"  # 샘플 데이터로 연습: "configs/sample_yolo.yaml"
cfg = load_config(CONFIG)
d = cfg["data"]

## 1. 데이터 읽기

In [ ]:
from src.dataset import build_class_map, load_records
from src.eda import dataset_summary, records_to_frames

records, categories = load_records(d["train_annotations"])
class_map = build_class_map(records, categories)
images_df, boxes_df = records_to_frames(records, class_map)
dataset_summary(images_df, boxes_df)

## 2. 클래스 분포
클래스마다 박스 수가 얼마나 다른가? 아주 적은 클래스는 증강·외부 데이터 보강 후보입니다.

In [ ]:
from src.eda import class_distribution
from src.visualize import plot_class_distribution

class_df = class_distribution(boxes_df)
plot_class_distribution(class_df)
class_df.head(10)

In [ ]:
class_df.tail(10)  # 가장 적은 클래스

**인사이트:** 

## 3. 이미지 크기와 이미지당 알약 수

In [ ]:
from src.eda import image_size_stats, objects_per_image

display(image_size_stats(images_df))
objects_per_image(images_df)

**인사이트:** 

## 4. 박스 크기·비율·위치
알약이 작으면 입력 해상도를 키우는 실험(D6), 위치가 몰려 있으면 이동·크롭 증강의 근거가 됩니다.

In [ ]:
from src.eda import bbox_stats
from src.visualize import plot_box_centers, plot_hist

display(bbox_stats(boxes_df))
plot_hist(boxes_df["rel_area"], "이미지 대비 박스 면적 비율", xlabel="rel_area")
plot_hist(boxes_df["aspect"], "박스 가로/세로 비율", xlabel="aspect")
plot_box_centers(boxes_df)

**인사이트:** 

## 5. 라벨 오류 의심 항목
대회 안내: 제공 데이터에 크고 작은 오류가 있습니다. 결과를 데이터 B에게 넘겨 정제(v1)에 반영합니다.

In [ ]:
from src.eda import find_label_issues

issues = find_label_issues(images_df, boxes_df, image_dir=d["train_images"])
display(issues["issue"].value_counts())
issues.head(20)

**인사이트:** 

## 6. 샘플 이미지

In [ ]:
from src.visualize import show_samples

show_samples(records, d["train_images"], class_map, n=6)

## 7. 테스트 이미지 확인
테스트에는 학습에 없는 클래스가 있습니다(대회 안내). 테스트 이미지 몇 장을 보고 학습 이미지와 촬영 조건이 비슷한지 봅니다.

In [ ]:
import matplotlib.pyplot as plt

from src.dataset import list_images, read_image_rgb

test_paths = list_images(d["test_images"])
print(f"테스트 이미지 {len(test_paths)}장")
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, p in zip(axes, test_paths[:4]):
    ax.imshow(read_image_rgb(p))
    ax.set_title(p.name)
    ax.axis("off")
plt.show()

## 정리
- 핵심 발견 3가지:
- 정제(데이터 B)에 넘길 항목:
- 실험 제안(모델 A·B):